# DDL: `dbspend360_pool_cloud_cost_explorer`

Per-pool EC2/EBS cloud cost, populated by the AWS cost explorer ETL's pool-tag
query (plan `plan_pool_pipeline_ec2_cost.md` §4.2 / CP5).

Idle/warm pool capacity is tagged `DatabricksInstancePoolId`, **not**
`ClusterId`, so it is invisible to `dbspend360_cloud_cost_explorer` (which
groups only by `ClusterId`). This dedicated table is the primary home of pool
EC2 cost. The explorer nets out any cost that *also* carries a `ClusterId`
(§4.3 guard), so pool and cluster cloud cost are disjoint — the tabs are
additive, not overlapping, for EC2.

Grain: `(instance_pool_id, cost_incurred_date, currency)`. On AWS this is a
single `cloud_cost` bucket (sum of the EC2 family); the `compute/storage/
network/other` segments are reserved but `NULL`. `idle_cloud_cost` /
`active_cloud_cost` are reserved for the future `instance_events`-based split
(§4.5) and stay `NULL` until that fast-follow lands.

**Widgets**
- `catalog` - target Unity Catalog name
- `schema`  - target schema name within `catalog`

In [ ]:
dbutils.widgets.text("catalog", "", "Catalog")
dbutils.widgets.text("schema", "", "Schema")

In [ ]:
catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()

if not catalog or not schema:
    raise ValueError("Both `catalog` and `schema` widgets must be set.")

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS ${catalog}.${schema}.dbspend360_pool_cloud_cost_explorer (
  instance_pool_id    STRING,
  cloud_cost          DOUBLE,
  compute_cost        DOUBLE,   -- NULL on AWS (single EC2/EBS bucket)
  storage_cost        DOUBLE,   -- NULL on AWS
  network_cost        DOUBLE,   -- NULL on AWS
  other_cost          DOUBLE,   -- NULL on AWS
  idle_cloud_cost     DOUBLE,   -- reserved (instance_events split, plan 4.5); NULL until then
  active_cloud_cost   DOUBLE,   -- reserved (instance_events split, plan 4.5); NULL until then
  currency            STRING,
  created_at          TIMESTAMP,
  updated_at          TIMESTAMP,
  cost_incurred_date  DATE
)
CLUSTER BY AUTO

In [ ]:
dbutils.notebook.exit(f"{catalog}.{schema}.dbspend360_pool_cloud_cost_explorer")